# Texture-model validation against Henderson et al. (2023), Fig. 3B, and comparison with the SP model

**Inputs**
- Henderson's original fits: `V2_drift/texturemodel/model_fits/MAGGIES/S0X/all_fit_params.npy` (her folder `texture_pyramid_ridge_4ori_4sf_pcaHL_fit_pRFs`)
- My whole-data TS fits: `texturemodel/code/run/fit_replication_all_subjects.sh` (PCA within Henderson's groups, 95%; OLS; all sessions)
- Noise ceiling and ROI labels: `V2_drift/nsd/rois/S{s}_noise_ceiling_avgreps.npy`, `S{s}_voxel_roi_info.npy`
- SP whole-data fits: `SP_Model/8_SP_wholedata_validation.ipynb`

**Metric (exactly as in Henderson's `Fig3_modelperf.ipynb`)**: mean validation R2 over ROI voxels with noise ceiling > 0.01; ROIs from the NSD pRF-mapping labels, with voxels overlapping FFA/OPA/PPA/RSC/EBA removed; bars = mean over 8 subjects, error = SEM.

My OLS fits contain a small number of numerically unstable voxels (R2 < -1, ~1.7%); they are excluded from the "clean" values and counted in the tables.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. S01 V2: voxel-by-voxel comparison

In [ ]:
# ==== S01 V2 voxel-by-voxel validation: Henderson vs. my TS replication (her pRFs) vs. (my pRFs) ====
import numpy as np
ROOT = "/content/drive/MyDrive/V2_drift/"
BASE = ROOT + "texturemodel/model_fits/"
RUN  = BASE + "S01/texture_pyramid_OLS_4ori_4sf_pcaHL_HendersonGroups_V2_pca95_use_{}_pRFs/{}/all_fit_params.npy"
fits = {
    "Henderson (ridge, her pRFs)": BASE + "MAGGIES/S01/all_fit_params.npy",
    "Mine (OLS, her pRFs)":        RUN.format("henderson_grid5", "Oct-15-2026_0912_22"),
    "Mine (OLS, my pRFs)":         RUN.format("nsd_grid5",       "Oct-19-2026_1201_46"),
}
def flat_index(v):
    v = v[0] if isinstance(v, (list, tuple)) else v
    return np.asarray(v).ravel().astype(int)
def col0(x):
    x = np.asarray(x); return x[:, 0] if x.ndim > 1 else x

D = {k: np.load(f, allow_pickle=True).item() for k, f in fits.items()}
vox = {k: flat_index(d["voxel_index"]) for k, d in D.items()}
common = vox["Mine (OLS, my pRFs)"]
for v in vox.values(): common = np.intersect1d(common, v)
pos = {k: np.array([np.where(v == c)[0][0] for c in common]) for k, v in vox.items()}
r2  = {k: col0(D[k]["val_r2"])[pos[k]] for k in D}
ok = np.all([np.isfinite(v) for v in r2.values()], axis=0)
rank = lambda x: np.argsort(np.argsort(x))
H = r2["Henderson (ridge, her pRFs)"][ok]
print(f"voxels in common: {len(common)}")
print(f"{'':30s} {'median':>8s} {'p90':>8s} {'Spearman vs her':>16s} {'R2<-1':>6s}")
for k, v in r2.items():
    v = v[ok]
    print(f"{k:30s} {np.median(v):8.4f} {np.percentile(v,90):8.4f} {np.corrcoef(rank(H), rank(v))[0,1]:16.3f} {(v<-1).sum():6d}")


## 2. Fig. 3B for V1-hV4: Henderson vs. my TS

In [ ]:
# ==== Henderson et al. (2023) Fig. 3B, reproduced for V1-hV4: her fits vs. my replication (my pRFs) ====
# Exactly her recipe (Fig3_modelperf.ipynb):
#   - noise ceiling from her precomputed files (nsd/rois/S{s}_noise_ceiling_avgreps.npy), keep NC > 0.01
#   - ROIs from the NSD pRF-mapping labels (non-Kastner), voxels overlapping FFA/OPA/PPA/RSC/EBA removed
#   - per subject x ROI: mean validation R2 (full model); bar = mean over subjects, error = SEM, dots = subjects
# My fits are matched to her voxels by whole-brain voxel index.
import glob, os, warnings, numpy as np, matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT      = "/content/drive/MyDrive/V2_drift/"
FITS      = ROOT + "texturemodel/model_fits/"
ROIS_DIR  = ROOT + "nsd/rois/"
PRF_MODEL = "nsd_grid5"            # my pRFs; use "henderson_grid5" for runs with her pRFs
MY_PATTERN = "texture_pyramid_OLS_4ori_4sf_pcaHL_HendersonGroups_{roi}_pca95_use_" + PRF_MODEL + "_pRFs"
ROI_CODES = {"V1": [1, 2], "V2": [3, 4], "V3": [5, 6], "hV4": [7]}
SUBJECTS  = range(1, 9)
NC_THRESH = 0.01
BLOWUP    = -1.0                   # my voxels with R2 below this are unstable OLS fits

def flat_index(v):
    v = v[0] if isinstance(v, (list, tuple)) else v
    return np.asarray(v).ravel().astype(int)
def col0(x):
    x = np.asarray(x); return x[:, 0] if x.ndim > 1 else x

def latest_valid_fit(subject, roi):
    runs = glob.glob(FITS + f"S{subject:02d}/" + MY_PATTERN.format(roi=roi) + "/*/all_fit_params.npy")
    for fn in sorted(runs, key=os.path.getmtime, reverse=True):
        d = np.load(fn, allow_pickle=True).item()
        if np.asarray(d["best_params"][1]).shape[1] > 60:      # skip pre-fix runs (54 features)
            return fn, d
    return None, None

nS, nR = len(SUBJECTS), len(ROI_CODES)
hers      = np.full((nS, nR), np.nan)
mine_all  = np.full((nS, nR), np.nan)     # her exact metric, all voxels
mine_clean= np.full((nS, nR), np.nan)     # same, excluding OLS blow-ups
n_vox     = np.zeros((nS, nR), int); n_blow = np.zeros((nS, nR), int); n_cov = np.zeros((nS, nR), int)

for si, s in enumerate(SUBJECTS):
    info = np.load(ROIS_DIR + f"S{s}_voxel_roi_info.npy", allow_pickle=True).item()
    vidx = flat_index(info["voxel_idx"])
    nc   = np.load(ROIS_DIR + f"S{s}_noise_ceiling_avgreps.npy") / 100
    assert len(nc) == len(vidx), f"S{s}: NC file length {len(nc)} != voxel list {len(vidx)}"
    ret, face, place, body = (np.asarray(info[k])[vidx] for k in
                              ["roi_labels_retino", "roi_labels_face", "roi_labels_place", "roi_labels_body"])
    categ = np.isin(face, [2, 3]) | np.isin(place, [1, 2, 3]) | np.isin(body, [1])   # FFA-1/2, OPA/PPA/RSC, EBA

    H = np.load(FITS + f"MAGGIES/S{s:02d}/all_fit_params.npy", allow_pickle=True).item()
    h_vox, h_r2 = flat_index(H["voxel_index"]), col0(H["val_r2"])
    h_lookup = dict(zip(h_vox, h_r2))

    for ri, (roi, codes) in enumerate(ROI_CODES.items()):
        sel = vidx[np.isin(ret, codes) & ~categ & (nc > NC_THRESH)]      # her voxels for this ROI
        n_vox[si, ri] = len(sel)
        hv = np.array([h_lookup.get(v, np.nan) for v in sel])
        hers[si, ri] = np.nanmean(hv)

        fn, M = latest_valid_fit(s, roi)
        if M is None:
            print(f"S{s} {roi}: no valid run of mine found"); continue
        m_lookup = dict(zip(flat_index(M["voxel_index"]), col0(M["val_r2"])))
        mv = np.array([m_lookup.get(v, np.nan) for v in sel])
        n_cov[si, ri] = np.isfinite(mv).sum()
        blow = mv < BLOWUP
        n_blow[si, ri] = blow.sum()
        mine_all[si, ri]   = np.nanmean(mv)
        mine_clean[si, ri] = np.nanmean(np.where(blow, np.nan, mv))

# ---------------- table ----------------
print(f"\n{'':10s} {'voxels':>7s} {'covered':>8s} {'Henderson':>10s} {'mine (all)':>11s} {'mine (clean)':>13s} {'blow-ups':>9s} {'% of hers':>10s}")
for si, s in enumerate(SUBJECTS):
    for ri, roi in enumerate(ROI_CODES):
        pct = 100 * mine_clean[si, ri] / hers[si, ri]
        print(f"S{s} {roi:6s} {n_vox[si,ri]:7d} {n_cov[si,ri]:8d} {hers[si,ri]:10.4f} {mine_all[si,ri]:11.4f} "
              f"{mine_clean[si,ri]:13.4f} {n_blow[si,ri]:9d} {pct:9.1f}%")
print("\nMEAN OVER SUBJECTS")
for ri, roi in enumerate(ROI_CODES):
    h, m = hers[:, ri], mine_clean[:, ri]
    print(f"{roi:4s} Henderson {np.nanmean(h):.4f} +/- {np.nanstd(h)/np.sqrt(np.sum(np.isfinite(h))):.4f} | "
          f"mine (clean) {np.nanmean(m):.4f} +/- {np.nanstd(m)/np.sqrt(np.sum(np.isfinite(m))):.4f} | "
          f"ratio {100*np.nanmean(m)/np.nanmean(h):.1f}% | total blow-ups excluded: {n_blow[:, ri].sum()}")

# ---------------- plot (her style) ----------------
series = [("Henderson et al. (2023)", hers, "#4a90c4"),
          (f"This study ({'my' if PRF_MODEL=='nsd_grid5' else 'her'} pRFs, OLS)", mine_clean, "#7a7a7a")]
x = np.arange(nR); width = 0.38
fig, ax = plt.subplots(figsize=(7, 4))
for k, (label, vals, color) in enumerate(series):
    pos = x + (k - 0.5) * width
    n = np.sum(np.isfinite(vals), axis=0)
    mean = np.nanmean(vals, axis=0)
    sem = np.nanstd(vals, axis=0) / np.sqrt(np.maximum(n, 1))
    ax.bar(pos, mean, width, color=color, label=label)
    ax.errorbar(pos, mean, yerr=sem, fmt="none", ecolor="black", lw=1.5)
    for si in range(nS):
        ax.plot(pos, vals[si], "o", ms=3.5, color="#cccccc", zorder=3)
ax.set_xticks(x); ax.set_xticklabels(list(ROI_CODES), rotation=45, fontsize=12)
ax.set_ylabel("R2", fontsize=12)
ax.set_ylim(0, 0.32)
ax.legend(frameon=False, fontsize=10)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.tight_layout()
out = ROOT + f"texturemodel/fig3_replication_V1-hV4_{PRF_MODEL}.png"
plt.savefig(out, dpi=300); plt.savefig(out.replace(".png", ".pdf")); plt.show()
print("saved:", out, "(+ .pdf)")
print("Note: my bars exclude voxels with R2 < -1 (unstable OLS fits); counts in the table above.")


## 3. Add the SP model (run after section 2)

In [ ]:
# ==== Fig. 3-style comparison: Henderson texture vs. my TS replication vs. my SP (whole-data fits) ====
# Run AFTER the Fig. 3 cell (uses hers, mine_clean, flat_index, ROIS_DIR, ROI_CODES, SUBJECTS, NC_THRESH, BLOWUP).
import numpy as np, matplotlib.pyplot as plt
SP_DIR = "/content/drive/MyDrive/V1_Drift/repDrift_expand/wholedata/"

sp_all   = np.full((len(SUBJECTS), len(ROI_CODES)), np.nan)
sp_clean = np.full_like(sp_all, np.nan)
sp_blow  = np.zeros(sp_all.shape, int); sp_cov = np.zeros(sp_all.shape, int); n_sel = np.zeros(sp_all.shape, int)

for si, s in enumerate(SUBJECTS):
    info = np.load(ROIS_DIR + f"S{s}_voxel_roi_info.npy", allow_pickle=True).item()
    vidx = flat_index(info["voxel_idx"]); nc = np.load(ROIS_DIR + f"S{s}_noise_ceiling_avgreps.npy") / 100
    ret, face, place, body = (np.asarray(info[k])[vidx] for k in
                              ["roi_labels_retino", "roi_labels_face", "roi_labels_place", "roi_labels_body"])
    categ = np.isin(face, [2, 3]) | np.isin(place, [1, 2, 3]) | np.isin(body, [1])
    SP = np.load(SP_DIR + f"SP_wholedata_sub{s}.npy", allow_pickle=True).item()
    for ri, (roi, codes) in enumerate(ROI_CODES.items()):
        sel = vidx[np.isin(ret, codes) & ~categ & (nc > NC_THRESH)]          # Henderson's voxels, as in Fig. 3
        lk = dict(zip(SP[roi]["voxel_index"], SP[roi]["val_r2"]))
        v = np.array([lk.get(x, np.nan) for x in sel])
        n_sel[si, ri], sp_cov[si, ri] = len(sel), np.isfinite(v).sum()
        blow = v < BLOWUP; sp_blow[si, ri] = blow.sum()
        sp_all[si, ri] = np.nanmean(v); sp_clean[si, ri] = np.nanmean(np.where(blow, np.nan, v))

print(f"{'':10s} {'voxels':>7s} {'SP covered':>11s} {'SP mean R2':>11s} {'SP clean':>9s} {'blow-ups':>9s}")
for si, s in enumerate(SUBJECTS):
    for ri, roi in enumerate(ROI_CODES):
        print(f"S{s} {roi:6s} {n_sel[si,ri]:7d} {sp_cov[si,ri]:11d} {sp_all[si,ri]:11.4f} {sp_clean[si,ri]:9.4f} {sp_blow[si,ri]:9d}")

sem = lambda a: np.nanstd(a, 0) / np.sqrt(np.sum(np.isfinite(a), 0))
print("\nMEAN OVER SUBJECTS (mean R2, NC > 0.01, Henderson's voxels)")
for ri, roi in enumerate(ROI_CODES):
    print(f"{roi:4s} Henderson TS {np.nanmean(hers[:,ri]):.4f}+/-{sem(hers)[ri]:.4f} | "
          f"my TS {np.nanmean(mine_clean[:,ri]):.4f}+/-{sem(mine_clean)[ri]:.4f} | "
          f"my SP {np.nanmean(sp_clean[:,ri]):.4f}+/-{sem(sp_clean)[ri]:.4f} | "
          f"SP/TS(mine) {100*np.nanmean(sp_clean[:,ri])/np.nanmean(mine_clean[:,ri]):.0f}%")

series = [("Henderson et al. (2023), TS", hers, "#4a90c4"),
          ("This study, TS", mine_clean, "#7a7a7a"),
          ("This study, SP (4 ori x 4 SF)", sp_clean, "#c9c9c9")]
x = np.arange(len(ROI_CODES)); w = 0.27
fig, ax = plt.subplots(figsize=(8, 4))
for k, (label, vals, color) in enumerate(series):
    p = x + (k - 1) * w
    m = np.nanmean(vals, 0)
    ax.bar(p, m, w, color=color, label=label, edgecolor="none")
    ax.errorbar(p, m, yerr=sem(vals), fmt="none", ecolor="black", lw=1.3)
    for si in range(vals.shape[0]):
        ax.plot(p, vals[si], "o", ms=3, color="#9a9a9a" if k == 2 else "#d0d0d0", zorder=3)
ax.set_xticks(x); ax.set_xticklabels(list(ROI_CODES), rotation=45, fontsize=12)
ax.set_ylabel("R2", fontsize=12); ax.set_ylim(0, 0.32); ax.legend(frameon=False, fontsize=9)
for side in ["top", "right"]: ax.spines[side].set_visible(False)
plt.tight_layout()
out = "/content/drive/MyDrive/V2_drift/texturemodel/fig3_TS_SP_V1-hV4.png"
plt.savefig(out, dpi=300); plt.savefig(out.replace(".png", ".pdf")); plt.show()
print("saved:", out, "(+ .pdf)")


## 4. All whole-data models on Henderson's voxels
Henderson TS (+ her Gabor if uploaded to `MAGGIES/S0X/gabor/`), my TS with Henderson's grouping using **her pRFs** and **my pRFs**, my TS pcaAll18 / pcaLL18, and SP. Missing models are skipped. Ends with a decomposition of the gap to Henderson into an OLS/other part (Henderson -> mine with her pRFs) and a pRF part (her pRFs -> my pRFs).

In [ ]:
# ==== Whole-data model comparison, V1-hV4, on Henderson's Fig. 3 voxels ====
# Metric as in Henderson et al. (2023) Fig. 3B: mean validation R2 over ROI voxels with noise ceiling > 0.01
# (pRF-mapping ROIs, category-ROI overlap removed); bar = mean over subjects, error = SEM, dots = subjects.
# Models (any that are missing are skipped and reported):
#   Henderson TS (ridge, her pRFs)        MAGGIES/S0X/all_fit_params.npy
#   Henderson Gabor 4ori x 4sf (optional) MAGGIES/S0X/gabor/all_fit_params.npy
#   my TS, Henderson groups, pca95        run/fit_replication_all_subjects.sh (PRF_MODEL=nsd_grid5 and =henderson_grid5)
#   my TS pcaAll18 / pcaLL18              run/fit_wholedata_fixed_n.sh
#   my SP (18 predictors)                 SP_Model/8_SP_wholedata_validation.ipynb
# My OLS fits: voxels with R2 < -1 are numerically unstable and excluded ("clean"); counts are printed.
import glob, os, warnings, numpy as np, matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT     = "/content/drive/MyDrive/V2_drift/"
FITS     = ROOT + "texturemodel/model_fits/"
ROIS_DIR = ROOT + "nsd/rois/"
SP_DIR   = "/content/drive/MyDrive/V1_Drift/repDrift_expand/wholedata/"
ROI_CODES = {"V1": [1, 2], "V2": [3, 4], "V3": [5, 6], "hV4": [7]}
SUBJECTS, NC_THRESH, BLOWUP = range(1, 9), 0.01, -1.0

def flat_index(v):
    v = v[0] if isinstance(v, (list, tuple)) else v
    return np.asarray(v).ravel().astype(int)
def col0(x):
    x = np.asarray(x); return x[:, 0] if x.ndim > 1 else x

def my_fit(subject, roi, folder, n_feat_ok):
    runs = glob.glob(FITS + f"S{subject:02d}/" + folder.format(roi=roi) + "/*/all_fit_params.npy")
    for fn in sorted(runs, key=os.path.getmtime, reverse=True):
        d = np.load(fn, allow_pickle=True).item()
        if n_feat_ok(np.asarray(d["best_params"][1]).shape[1]):
            return dict(zip(flat_index(d["voxel_index"]), col0(d["val_r2"])))
    return None

def hers_fit(subject, roi, sub=""):
    fn = FITS + f"MAGGIES/S{subject:02d}/{sub}all_fit_params.npy"
    if not os.path.exists(fn): return None
    d = np.load(fn, allow_pickle=True).item()
    return dict(zip(flat_index(d["voxel_index"]), col0(d["val_r2"])))

def sp_fit(subject, roi):
    fn = SP_DIR + f"SP_wholedata_sub{subject}.npy"
    if not os.path.exists(fn): return None
    d = np.load(fn, allow_pickle=True).item()[roi]
    return dict(zip(d["voxel_index"], d["val_r2"]))

MODELS = {   # name: (loader, color, is_mine)
    "Henderson TS (ridge)":     (lambda s, r: hers_fit(s, r), "#4a90c4", False),
    "Henderson Gabor (ridge)":  (lambda s, r: hers_fit(s, r, "gabor/"), "#a8c8e4", False),
    "TS Henderson groups, her pRFs": (lambda s, r: my_fit(s, r, "texture_pyramid_OLS_4ori_4sf_pcaHL_HendersonGroups_{roi}_pca95_use_henderson_grid5_pRFs", lambda n: n > 60), "#2f2f2f", True),
    "TS Henderson groups, 95%": (lambda s, r: my_fit(s, r, "texture_pyramid_OLS_4ori_4sf_pcaHL_HendersonGroups_{roi}_pca95_use_nsd_grid5_pRFs", lambda n: n > 60), "#555555", True),
    "TS pcaAll, 18 PCs":        (lambda s, r: my_fit(s, r, "texture_pyramid_OLS_4ori_4sf_pcaAll_{roi}_n18_use_nsd_grid5_pRFs", lambda n: n == 18), "#888888", True),
    "TS 1 PC/group (58)": (lambda s, r: my_fit(s, r, "texture_pyramid_OLS_4ori_4sf_pcaHL_HendersonGroups_{roi}_k1_use_nsd_grid5_pRFs", lambda n: n == 58), "#9a9a9a", True),
    "TS 2 PCs/group (67)": (lambda s, r: my_fit(s, r, "texture_pyramid_OLS_4ori_4sf_pcaHL_HendersonGroups_{roi}_k2_use_nsd_grid5_pRFs", lambda n: n == 67), "#7a7a7a", True),
    "TS 2 PCs/group, z-scored (67)": (lambda s, r: my_fit(s, r, "texture_pyramid_OLS_4ori_4sf_pcaHL_HendersonGroups_{roi}_k2z_use_nsd_grid5_pRFs", lambda n: n == 67), "#6f6f6f", True),
    "TS pcaLL, 18 PCs":         (lambda s, r: my_fit(s, r, "texture_pyramid_OLS_4ori_4sf_pcaLL_{roi}_n18_use_nsd_grid5_pRFs", lambda n: n == 18), "#b0b0b0", True),
    "SP 4ori x 4sf (18)":       (sp_fit, "#d6d6d6", True),
}

nS, nR = len(SUBJECTS), len(ROI_CODES)
vals  = {m: np.full((nS, nR), np.nan) for m in MODELS}
blows = {m: np.zeros((nS, nR), int) for m in MODELS}
cover = {m: np.zeros((nS, nR), int) for m in MODELS}
n_sel = np.zeros((nS, nR), int)

for si, s in enumerate(SUBJECTS):
    info = np.load(ROIS_DIR + f"S{s}_voxel_roi_info.npy", allow_pickle=True).item()
    vidx = flat_index(info["voxel_idx"]); nc = np.load(ROIS_DIR + f"S{s}_noise_ceiling_avgreps.npy") / 100
    ret, face, place, body = (np.asarray(info[k])[vidx] for k in
                              ["roi_labels_retino", "roi_labels_face", "roi_labels_place", "roi_labels_body"])
    categ = np.isin(face, [2, 3]) | np.isin(place, [1, 2, 3]) | np.isin(body, [1])
    for ri, (roi, codes) in enumerate(ROI_CODES.items()):
        sel = vidx[np.isin(ret, codes) & ~categ & (nc > NC_THRESH)]
        n_sel[si, ri] = len(sel)
        for m, (load, _, mine) in MODELS.items():
            lk = load(s, roi)
            if lk is None: continue
            v = np.array([lk.get(x, np.nan) for x in sel], dtype=float)
            cover[m][si, ri] = np.isfinite(v).sum()
            if mine:
                blows[m][si, ri] = np.sum(v < BLOWUP); v[v < BLOWUP] = np.nan
            vals[m][si, ri] = np.nanmean(v)

present = [m for m in MODELS if np.isfinite(vals[m]).any()]
missing = [m for m in MODELS if m not in present]
if missing: print("not found (skipped):", missing)

sem = lambda a: np.nanstd(a, 0) / np.sqrt(np.maximum(np.sum(np.isfinite(a), 0), 1))
print(f"\n{'model':28s} " + "  ".join(f"{r:>15s}" for r in ROI_CODES) + "   n_subj  excluded(R2<-1)  coverage")
for m in present:
    a = vals[m]
    cells = "  ".join(f"{np.nanmean(a[:,ri]):.4f}+/-{sem(a)[ri]:.4f}" for ri in range(nR))
    cov = 100 * cover[m].sum() / np.where(np.isfinite(a), n_sel, 0).sum()
    print(f"{m:28s} {cells}   {int(np.sum(np.isfinite(a).all(1))):6d}  {blows[m].sum():15d}  {cov:7.1f}%")

x = np.arange(nR); w = 0.8 / len(present)
fig, ax = plt.subplots(figsize=(10, 4))
for k, m in enumerate(present):
    a = vals[m]; p = x - 0.4 + w * (k + 0.5); mu = np.nanmean(a, 0)
    ax.bar(p, mu, w, color=MODELS[m][1], label=m, edgecolor="none")
    ax.errorbar(p, mu, yerr=sem(a), fmt="none", ecolor="black", lw=1.2)
    for si in range(nS):
        ax.plot(p, a[si], "o", ms=2.5, color="#9a9a9a", zorder=3)
ax.set_xticks(x); ax.set_xticklabels(list(ROI_CODES), fontsize=12)
ax.set_ylabel("validation R2 (whole-data fit)", fontsize=11); ax.set_ylim(0, 0.32)
ax.legend(frameon=False, fontsize=8, ncol=2)
for side in ["top", "right"]: ax.spines[side].set_visible(False)
plt.tight_layout()
out = ROOT + "texturemodel/wholedata_models_V1-hV4.png"
plt.savefig(out, dpi=300); plt.savefig(out.replace(".png", ".pdf")); plt.show()
print("saved:", out, "(+ .pdf)")

# ---- where does the gap to Henderson come from? (subjects that have all three fits) ----
H, Mh, Mn = "Henderson TS (ridge)", "TS Henderson groups, her pRFs", "TS Henderson groups, 95%"
if all(m in present for m in (H, Mh, Mn)):
    print("\nGAP DECOMPOSITION (same subjects in all three; % of Henderson's R2)")
    print(f"{'ROI':5s} {'n_subj':>6s} {'Henderson':>10s} {'mine,her pRF':>13s} {'mine,my pRF':>12s} {'OLS & rest':>11s} {'pRF':>7s}")
    for ri, roi in enumerate(ROI_CODES):
        ok = np.isfinite(vals[H][:, ri]) & np.isfinite(vals[Mh][:, ri]) & np.isfinite(vals[Mn][:, ri])
        if not ok.any(): continue
        h, a, b = (np.mean(vals[m][ok, ri]) for m in (H, Mh, Mn))
        print(f"{roi:5s} {ok.sum():6d} {h:10.4f} {a:13.4f} {b:12.4f} {100*(h-a)/h:10.1f}% {100*(a-b)/h:6.1f}%")
    print("OLS & rest = Henderson -> mine with her pRFs ; pRF = mine with her pRFs -> mine with my pRFs")


## 5. Utility: create Henderson's pRF index files for all subjects x ROIs
Needed before running `run/fit_replication_all_subjects.sh` with `PRF_MODEL=henderson_grid5`. Checks that it reproduces the existing S01 V2 file first.

In [ ]:
# ==== Create Henderson's best-pRF index files for every subject x ROI (needed for PRF_MODEL=henderson_grid5) ====
# Takes each voxel's best pRF from her fits (best_params[5]), matched by voxel index to the voxels that
# fit_model.py selects for that subject/ROI. First re-creates S01 V2 and checks it equals the existing file.
import os, sys, numpy as np
ROOT = "/content/drive/MyDrive/V2_drift/"
for p in [ROOT + "texturemodel/", ROOT + "texturemodel/code/", ROOT + "texturemodel/code/utils"]:
    if p not in sys.path: sys.path.append(p)
from utils import roi_utils

MAGGIES = ROOT + "texturemodel/model_fits/MAGGIES/"
PRF_DIR = ROOT + "prf_indices/"
ROI_CODES_FIT = {"V1": [1, 2], "V2": [3, 4], "V3": [5, 6], "hV4": [7]}   # same as fit_model.py

def _flat(v):
    v = v[0] if isinstance(v, (list, tuple)) else v
    return np.asarray(v).ravel().astype(int)

def her_prfs_for(subject, roi):
    voxel_mask, _, voxel_roi, _, _ = roi_utils.get_voxel_roi_info(subject, True)       # as in fit_model.py
    my_vox = np.where(voxel_mask & np.isin(voxel_roi[0], ROI_CODES_FIT[roi]))[0]
    H = np.load(MAGGIES + f"S{subject:02d}/all_fit_params.npy", allow_pickle=True).item()
    h_prf = np.asarray(H["best_params"][5]); h_prf = h_prf[:, 0] if h_prf.ndim > 1 else h_prf
    lookup = dict(zip(_flat(H["voxel_index"]), h_prf.astype(int)))
    missing = [v for v in my_vox if v not in lookup]
    if missing:
        raise RuntimeError(f"S{subject} {roi}: {len(missing)} of {len(my_vox)} voxels not in her fit")
    return np.array([lookup[v] for v in my_vox], dtype=int)[:, None]

existing = np.load(PRF_DIR + "subj01_henderson_best_prf_indices_V2.npy", allow_pickle=True).squeeze().astype(int)
new = her_prfs_for(1, "V2").squeeze()
# Grid 5 contains a few near-duplicate entries (identical x, y, sigma), so two different indices can be the same pRF.
from model_fitting import initialize_fitting
g = initialize_fitting.get_prf_models(which_grid=5)
diff = existing != new
same_prf = np.allclose(g[existing[diff]], g[new[diff]], atol=1e-6)
print(f"S01 V2 check: {np.sum(~diff)}/{len(new)} identical indices; "
      f"{diff.sum()} differ but point to identical pRF parameters: {same_prf}")
assert len(existing) == len(new) and same_prf, "Real mismatch with the existing S01 V2 file -- stop and investigate"

for s in range(1, 9):
    for roi in ROI_CODES_FIT:
        fn = PRF_DIR + f"subj{s:02d}_henderson_best_prf_indices_{roi}.npy"
        if os.path.exists(fn):
            print(f"exists, kept: {os.path.basename(fn)}"); continue
        try:
            idx = her_prfs_for(s, roi); np.save(fn, idx)
            print(f"saved: {os.path.basename(fn)}  {idx.shape}")
        except Exception as e:
            print(f"FAILED S{s} {roi}: {e}")
